In [ ]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

library(dplyr)
library(WGCNA)
library(data.table)

source("/mnt/lareaulab/reliscu/code/FindModules/FindModules.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: dynamicTreeCut

Loading required package: fastcluster


Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last



Attaching package: ‘flashClust’


The following object is masked from ‘package:fastcluster’:

    hclust


The following object is masked from ‘package:stats’:

    hclust



Attaching package: ‘svMisc’


The following object is masked from ‘package:utils’:

    ?


Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:



Allowing multi-threading with up to 48 threads.


In [ ]:
data_source <- "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed"
expr <- fread("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-58-02/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed.csv", data.table=FALSE)
colnames(expr)[1] <- "Gene"

In [ ]:
marker_genes <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

sampleinfo <- read.csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv")

In [ ]:
# Remove genes from unwanted modules

top_mods_df <- read.csv("data/EA/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.95_subsetCutoff68.086_Modules_Curated_EA_top_Qval_mods.csv")

mask <- top_mods_df$Cell_type == "OPC"
module <- top_mods_df[mask, 'Module']
kme_df <- fread(top_mods_df[mask, 'kME_path'], data.table=FALSE)
mod1_genes <- kme_df[kme_df[,grep("TopModPosFDR", colnames(kme_df))] %in% module, 'Gene']
length(mod1_genes)

mask <- top_mods_df$Cell_type == "Astro"
module <- top_mods_df[mask, 'Module']
kme_df <- fread(top_mods_df[mask, 'kME_path'], data.table=FALSE)
mod_kme <- kme_df[,c("Gene", paste0("kME", module))]
mod_kme <- mod_kme[order(-mod_kme[,2]),]
mod2_genes <- mod_kme$Gene[mod_kme[,2] >= .75]
length(mod2_genes)

module <- "slateblue2"
kme_df <- fread("GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.95_subsetCutoff68.086_Modules/Bicor-None_signum0.763_minSize6_merge_ME_0.95_15915/kME_table_08-37-20.csv", data.table=FALSE)
mod_kme <- kme_df[,c("Gene", paste0("kME", module))]
mod_kme <- mod_kme[order(-mod_kme[,2]),]
mod3_genes <- mod_kme$Gene[mod_kme[,2] >= .75]
length(mod3_genes)

module <- "orange4"
kme_df <- fread("GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.95_subsetCutoff68.086_Modules/Bicor-None_signum0.75_minSize6_merge_ME_0.95_15915/kME_table_08-03-53.csv", data.table=FALSE)
mod_kme <- kme_df[,c("Gene", paste0("kME", module))]
mod_kme <- mod_kme[order(-mod_kme[,2]),]
mod4_genes <- mod_kme$Gene[mod_kme[,2] >= .75]
length(mod4_genes)

# ...but keep markers of desired cell type

mod_genes <- unique(c(mod1_genes, mod2_genes, mod3_genes, mod4_genes))
mod_genes <- mod_genes[!mod_genes %in% unname(unlist(marker_genes))] # mod_genes[!mod_genes %in% marker_genes[["OPC"]]]
length(mod_genes)

not_mod_genes <- !(expr[,1] %in% mod_genes)

In [ ]:
# Subset to genes in the top X percentile

not_ribo_genes <- !grepl("^RP", expr[,1])
not_mito_genes <- !grepl("^MT−", expr[,1])

prob <- .6
mean_expr <- rowMeans(expr[,-1])

min_cutoff <- unname(quantile(mean_expr, prob))
print(paste("quantile(mean_expr, prob):", round(min_cutoff, 3)))

subset <- (mean_expr >= min_cutoff) & not_mod_genes & not_ribo_genes & not_mito_genes 
sum(subset)

In [ ]:
# Order samples by covariates of interest

sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[sampleinfo[,1] %in% colnames(expr),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))
sampleinfo$SAMPID <- make.names(sampleinfo$SAMPID)
sampleinfo <- sampleinfo %>% arrange(Mean_age)
    
expr <- expr[, c(1, match(sampleinfo[,1], colnames(expr)[-1]) + 1)]
all.equal(sampleinfo[,1], colnames(expr)[-1])

In [ ]:
samplegroups <- as.factor(sampleinfo$AGE)
merge.param <- 0.93
projectname <- paste0(data_source, "_mergeParam", merge.param, "_subsetCutoff", round(min_cutoff, 3))

In [ ]:
FindModules(
  projectname=projectname,
  expr=expr,
  geneinfo=1,
  sampleindex=2:ncol(expr),
  samplegroups=samplegroups,
  subset=subset,
  simMat=NULL,
  saveSimMat=FALSE,
  simType="Bicor",
  beta=1,
  overlapType="None",
  TOtype="signed",
  TOdenom="min",
  MIestimator="mi.mm",
  MIdisc="equalfreq",
  signumType="rel",
  iterate=TRUE,
  signumvec=rev(c(.95,.94,.93,.92,.91,.9)), 
  minsizevec=c(4,5,6,8,10), 
  signum=NULL,
  minSize=NULL,
  merge.by="ME",
  merge.param=merge.param,
  export.merge.comp=T,
  ZNCcut=2,
  calcSW=FALSE,
  loadTree=FALSE,
  writeKME=TRUE,
  calcBigModStat=FALSE,
  writeModSnap=TRUE
)